In [26]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Set global Matplotlib parameters
plt.rcParams["font.family"] = "serif"
plt.rcParams["text.usetex"] = True
plt.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"
plt.rcParams["figure.dpi"] = 1200
plt.rcParams["font.size"] = 20
plt.rcParams["axes.labelsize"] = 20
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["legend.fontsize"] = 20
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.major.size"] = 5.0
plt.rcParams["xtick.minor.size"] = 3.0
plt.rcParams["ytick.major.size"] = 5.0
plt.rcParams["ytick.minor.size"] = 3.0
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["legend.handlelength"] = 2.0

# --- Define color mapping for materials ---
COLOR_MAP = {
    'Ag-aSi': '#FF0000',    # Red
    'AlOx-HfO2': '#70AD47', # Green
    'EpiRAM': '#4472C4',    # Blue
    'TaOx-HfOx': '#ED7D31'  # Orange
}

# Device Performance Improvement Attributed to Algorithmic Contributions

In [27]:
# =============================================================================
# Constants
# =============================================================================
ITERATIONS = [1, 3, 5, 7, 9, 11, 13]
MATERIALS = ['Ag-aSi', 'AlOx-HfO2', 'EpiRAM', 'TaOx-HfOx']
QUANTITIES = ['L2_norm_Error', 'Loo_norm_Error', 'writeLatency_mean', 'writeEnergy_mean']
LABELS = [
    r"$\epsilon_{\| \cdot \|_2}(\text{rel})$",
    r"$\epsilon_{\| \cdot \|_\infty}(\text{rel})$",
    r"$\text{L}_{\text{write}} [\mathrm{s}]$",
    r"$\text{E}_{\text{write}} [J]$"
]
MATRIX_MAPPING = {
    'exp1': 'bcsstk02',
    'exp2': 'Iperturb',
    'exp3': 'bcsstk02',
    'exp4': 'Iperturb'
}

# Set up output directory
output_dir = Path('./documents/figures')
output_dir.mkdir(parents=True, exist_ok=True)

# =============================================================================
# Plotting Function
# =============================================================================
def trend_plot(data, experiment_ids, file_suffix):
    """
    Plot data for given experiments with specified stylistic elements.
    
    Parameters:
    - data: DataFrame containing the data
    - experiment_ids: List of experiment IDs to filter (e.g., ['exp1', 'exp2'])
    - file_suffix: Suffix for output file names (e.g., '_EC1')
    """
    # --- Filter data ---
    filtered_data = data[
        (data['iteration'].isin(ITERATIONS)) &
        (data['material'].isin(MATERIALS)) &
        (data['exp_id'].isin(experiment_ids))
    ]
    
    # --- Group data by material, experiment, and iteration ---
    grouped_data = filtered_data.groupby(['material', 'exp_id', 'iteration'])[QUANTITIES].mean().reset_index()
    
    # ---  Plot for each quantity and experiment ---
    for quantity, label in zip(QUANTITIES, LABELS):
        for exp_id in experiment_ids:
            matrix_label = MATRIX_MAPPING.get(exp_id, 'unknown')
            
            # Create figure
            fig, ax = plt.subplots(figsize=(10, 6))
            
            # Plot each material's data
            for material in MATERIALS:
                material_data = grouped_data[
                    (grouped_data["material"] == material) &
                    (grouped_data["exp_id"] == exp_id)
                ].sort_values('iteration')
                
                ax.plot(
                    material_data["iteration"] - 1,  # Shift iterations for plotting
                    material_data[quantity],
                    marker="o",
                    linestyle='-',
                    markersize=5,
                    color=COLOR_MAP[material],
                    label=material
                )
            
            # Configure axes
            ax.set_xlim(-0.5, 12.5)  # Buffer for markers
            ax.set_xlabel("Iteration")
            ax.set_ylabel(label)
            ax.set_yscale("log")
            ax.xaxis.set_major_locator(plt.MultipleLocator(1))
            
            # Place the legend at the bottom center, outside the main plot
            ax.legend(title="Material", loc="upper center",
            bbox_to_anchor=(0.5, -0.15),  # move below the axes and horizontally center
            ncol=len(MATERIALS),          # one column per material
            borderaxespad=0.0)            # no padding around the legend box
            
            # Adjust layout so the bottom legend is not cut off
            plt.tight_layout(rect=[0, 0, 1, 1.0])
            
            # Save plot
            output_filename = output_dir / f"{matrix_label}_{quantity}{file_suffix}.pdf"
            plt.savefig(output_filename, format='pdf')
            plt.close(fig)

In [28]:
# --- Load data once ---
data = pd.read_csv('../data/varied_iters.csv')
# --- Generate plots for both experiment sets ---
trend_plot(data, ['exp1', 'exp2'], '_EC1')
trend_plot(data, ['exp3', 'exp4'], '_EC0')

# Weak-Scaling

In [47]:
# =============================================================================
# Constants
# =============================================================================
ITERATIONS = [21]
MATERIALS = ['Ag-aSi', 'AlOx-HfO2', 'EpiRAM', 'TaOx-HfOx']
EXPERIMENTS = ['exp1', 'exp2', 'exp3', 'exp4', 'exp5', 'exp6']
QUANTITIES = ['L2_norm_Error', 'Loo_norm_Error', 'writeLatency_mean', 'writeEnergy_mean']
LABELS = [
    r"$\epsilon_{\| \cdot \|_2}(\text{rel})$",
    r"$\epsilon_{\| \cdot \|_\infty}(\text{rel})$",
    r"$\text{L}_{\text{write}} [\mathrm{s}]$",
    r"$\text{E}_{\text{write}} [J]$"
]
CELL_SIZES = ["$32\\times32$", "$64\\times64$", "$128\\times128$",
              "$256\\times256$", "$512\\times512$", "$1024\\times1024$"]


# =============================================================================
# Plotting Function
# =============================================================================
def boxplot(grouped_data, raw_data, quantity, ylabel, output_filename, cell_sizes=CELL_SIZES):
    """
    Plot the specified quantity as a function of cell size for each material.
    Each plot includes:
      - A trend line connecting the mean values (from grouped data)
      - Boxplots showing the distribution of raw data for each experiment
      - Custom x-tick labels, a log-scaled y-axis, a legend, and a shaded region.

    Parameters:
        grouped_data (pd.DataFrame): DataFrame grouped by material, exp_id, and iteration with mean values.
        raw_data (pd.DataFrame): Raw filtered DataFrame used for plotting the boxplots.
        quantity (str): Column name for the quantity to plot.
        ylabel (str): Label for the y-axis.
        output_filename (str): Path to save the output PDF.
        cell_sizes (list of str): Labels for the x-axis corresponding to each experiment.
    """
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # Use the constant order from EXPERIMENTS and corresponding cell_sizes
    exp_ids = EXPERIMENTS
    
    box_width = 0.15  # Width for the boxplots
    
    # Loop through each material and gather data per experiment
    for material in MATERIALS:
        mat_color = COLOR_MAP[material]
        exp_indices = []  # x-positions for the trend line points
        trend_values = [] # Mean values (from grouped data)
        all_box_data = [] # Raw data arrays for boxplots
        box_positions = []  # x-positions for each boxplot
        
        for idx, (exp_id, cell_label) in enumerate(zip(exp_ids, cell_sizes)):
            # Get the mean (grouped) value for the current material and experiment
            grouped_point = grouped_data[
                (grouped_data["material"] == material) & (grouped_data["exp_id"] == exp_id)
            ]
            # Get raw values for boxplot from the ungrouped filtered data
            raw_values = raw_data[
                (raw_data["material"] == material) & (raw_data["exp_id"] == exp_id)
            ][quantity].values
            
            if len(raw_values) > 0 and not grouped_point.empty:
                exp_indices.append(idx)
                trend_values.append(grouped_point[quantity].iloc[0])
                box_positions.append(idx)
                all_box_data.append(raw_values)
        
        # Plot the trend line for the material
        ax.plot(
            exp_indices,
            trend_values,
            marker="o",
            linestyle='-',
            markersize=8,
            color=mat_color,
            label=material,
            zorder=3
        )
        
        # Plot the boxplots for the material
        ax.boxplot(
            all_box_data,
            positions=box_positions,
            widths=box_width,
            patch_artist=True,
            showfliers=False,
            boxprops=dict(facecolor=mat_color, alpha=0.4),
            medianprops=dict(color=mat_color, linewidth=2),
            whiskerprops=dict(color=mat_color, linestyle='--'),
            capprops=dict(color=mat_color),
            zorder=2
        )
    
    # Configure x-axis with cell size labels
    ax.set_xticks(range(len(cell_sizes)))
    ax.set_xticklabels(cell_sizes, rotation=45, ha='right')
    ax.set_xlim(-0.5, len(cell_sizes) - 0.5)
    
    # Set y-axis to logarithmic scale and add labels
    ax.set_yscale("log")
    ax.set_xlabel("MCA Size", labelpad=12)
    ax.set_ylabel(ylabel, labelpad=12)
    
    # Build a custom legend for the materials
    handles = [
        plt.Line2D([0], [0], color=COLOR_MAP[m], marker='o', linestyle='-', markersize=8, label=m)
        for m in MATERIALS
    ]

    # Place the legend at the bottom center, outside the main plot
    ax.legend(title="Material", loc="upper center",
    bbox_to_anchor=(0.5, -0.25),  # move below the axes and horizontally center
    ncol=len(MATERIALS),          # one column per material
    borderaxespad=0.0)            # no padding around the legend box
    
    # Add shaded region to indicate the virtualization zone
    shading_start = CELL_SIZES[0]  # Starting at the first cell size ("$32\\times32$")
    shading_start_idx = cell_sizes.index(shading_start)
    ax.axvspan(
        xmin=shading_start_idx, 
        xmax=len(cell_sizes) - 3,
        color='yellow',
        alpha=0.15
    )
    
    # Add text annotation for virtualization within the plot area
    ax.text(
        1.5, 0.95, "Virtualization",
        transform=ax.get_xaxis_transform(),
        bbox=dict(facecolor='white', alpha=1.0, edgecolor='black'),
        ha='center',
        va='center')
    
    plt.tight_layout()
    plt.savefig(output_filename, bbox_inches='tight')
    plt.close(fig)

In [48]:
# --- Load the CSV file ---
data = pd.read_csv('../data/commercialized.csv')

# --- Set up the output directory ---
OUTPUT_DIR = "../figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Filter the dataset
filtered_data = data[
    data['iteration'].isin(ITERATIONS) &
    data['material'].isin(MATERIALS) &
    data['exp_id'].isin(EXPERIMENTS)
]

# Group the data by material, exp_id, and iteration to compute mean values
grouped_data = filtered_data.groupby(['material', 'exp_id', 'iteration'])[QUANTITIES].mean().reset_index()

# Loop over each quantity and generate its corresponding plot
for quantity, label in zip(QUANTITIES, LABELS):
    output_filename = os.path.join(OUTPUT_DIR, f"{quantity}_with_distributions.pdf")
    boxplot(grouped_data, filtered_data, quantity, label, output_filename)

# Strong Scaling

In [63]:
# =============================================================================
# Constants
# =============================================================================
ITERATIONS = [21]
MATERIALS = ['Ag-aSi', 'AlOx-HfO2', 'EpiRAM', 'TaOx-HfOx']
EXPERIMENTS = ['exp1', 'exp2', 'exp3', 'exp4', 'exp5', 'exp6']
QUANTITIES = ["writeLatency_mean", "writeEnergy_mean", "Loo_norm_Error", "L2_norm_Error"]
LABELS = [
    r"$\epsilon_{\| \cdot \|_2}(\text{rel})$",
    r"$\epsilon_{\| \cdot \|_\infty}(\text{rel})$",
    r"$\text{L}_{\text{write}} [\mathrm{s}]$",
    r"$\text{E}_{\text{write}} [J]$"
]
MATRIX_SIZES = [
    "$2903\\times2903$", "$4960\\times4960$", "$8127\\times8127$",
    "$16129\\times16129$", "$32226\\times32226$", "$65025\\times65025$"
]

# --- Plot configuration constants ---
SHADING_START = "$16129\\times16129$"  # Matrix size at which normalization/shading starts
XTICK_ROTATION = 45
LEGEND_NCOL = 4

# =============================================================================
# Plotting Function
# =============================================================================
def plot_strong_scaling(grouped_data, quantities, labels, matrix_sizes, output_dir):
    """
    Plot strong scaling data for each quantity and material type.

    For each quantity, the function:
      - Extracts the final iteration value (highest iteration) per material and experiment.
      - Maps experiments to their corresponding matrix sizes for the x-axis.
      - Plots a trend line for each material.
      - For latency and energy quantities, plots an additional normalized (divided by 16)
        trend line starting from the defined shading index.
      - Adds a shaded region to indicate the normalization zone.
      - Configures the axis, grid, and legend before saving the figure.

    Parameters:
        grouped_data (pd.DataFrame): Data grouped by material, exp_id, and iteration with mean values.
        quantities (list of str): List of column names to plot.
        labels (list of str): Corresponding y-axis labels.
        matrix_sizes (list of str): Matrix size labels for the x-axis.
        output_dir (str): Directory path for saving the output plots.
    """
    # Unique materials and experiments from the data
    materials = grouped_data["material"].unique()
    experiments = grouped_data["exp_id"].unique()
    
    # Create a mapping between experiment IDs and matrix sizes (using fixed EXPERIMENTS order)
    exp_size_map = dict(zip(sorted(EXPERIMENTS), matrix_sizes))
    
    for quantity, ylabel in zip(quantities, labels):
        fig, ax = plt.subplots(figsize=(14, 12))
        
        # For each (material, exp_id) combination, select the record with the highest iteration
        final_data = (
            grouped_data
            .groupby(['material', 'exp_id'])
            .apply(lambda df: df.nlargest(1, 'iteration'))
            .reset_index(drop=True)
        )
        # Map each experiment to its corresponding matrix size
        final_data['matrix_size'] = final_data['exp_id'].map(exp_size_map)
        
        # Plot data for each material
        for material in materials:
            material_df = final_data[final_data['material'] == material]
            if not material_df.empty:
                # Retrieve x-axis values (matrix sizes) and y-values for the current quantity
                x_values = material_df['exp_id'].map(exp_size_map).values
                y_values = material_df[quantity].values
                
                # Plot the original trend line
                line_obj = ax.plot(
                    x_values,
                    y_values,
                    marker="o",
                    linestyle='-',
                    markersize=8,
                    label=material,
                    linewidth=2
                )
                line_color = line_obj[0].get_color()
                
                # For write latency and energy, plot an additional normalized trend line
                if quantity in ["writeLatency_mean", "writeEnergy_mean"]:
                    shading_start_idx = matrix_sizes.index(SHADING_START)
                    adjusted_values = y_values.copy()
                    # Divide values by 16 starting from the shading start index
                    adjusted_values[shading_start_idx:] = adjusted_values[shading_start_idx:] / 16
                    ax.plot(
                        x_values,
                        adjusted_values,
                        marker="o",
                        linestyle='--',  # Dashed line distinguishes normalization
                        markersize=8,
                        color=line_color,
                        linewidth=2,
                        alpha=0.7
                    )
        
        # Add a shaded region to indicate the normalization area
        shading_start_idx = matrix_sizes.index(SHADING_START)
        ax.axvspan(
            xmin=shading_start_idx, 
            xmax=len(matrix_sizes) - 1,
            color='yellow',
            alpha=0.25
        )
        
        # Add a text box inside the figure to indicate the virtualization region.
        # Place the text box at x = 4.0 (center of the virtualization zone)
        ax.text(4.0, 0.95, "Virtualization", transform=ax.get_xaxis_transform(),
                bbox=dict(facecolor='white', alpha=1.0, edgecolor='black'),
                ha='center', va='center', fontsize=16)
                
        # Configure x-axis with the matrix size labels
        ax.set_xticks(matrix_sizes)
        ax.set_xticklabels(matrix_sizes, rotation=XTICK_ROTATION, ha='right')
        ax.set_xlabel("Matrix Size", fontsize=16)
        ax.set_ylabel(ylabel, fontsize=16)
        ax.set_yscale('log')
        
        # Configure legend
        ax.legend(
            title="Material",
            loc='upper center',
            bbox_to_anchor=(0.5, -0.2),
            ncol=min(LEGEND_NCOL, len(materials) * 2),
            frameon=True,
            fontsize=16
        )
        
        plt.tight_layout(rect=[0, 0.02, 1, 0.95])
        output_filename = os.path.join(output_dir, f"{quantity}_by_matrixSizes.pdf")
        plt.savefig(output_filename, bbox_inches='tight')
        plt.close(fig)



In [64]:
# Load the CSV file
data = pd.read_csv('../data/strongScaling.csv')

# Filter the dataset by iteration, material, and experiment
filtered_data = data[
    data['iteration'].isin(ITERATIONS) &
    data['material'].isin(MATERIALS) &
    data['exp_id'].isin(EXPERIMENTS)
]

# Group the data by material, exp_id, and iteration; compute the mean values
grouped_data = filtered_data.groupby(['material', 'exp_id', 'iteration'])[QUANTITIES].mean().reset_index()

# Execute the strong scaling plotting function
plot_strong_scaling(
    grouped_data=grouped_data,
    quantities=QUANTITIES,
    labels=LABELS,
    matrix_sizes=MATRIX_SIZES,
    output_dir=OUTPUT_DIR
)

C:\Users\quang\AppData\Local\Temp\ipykernel_8756\3282043579.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df.nlargest(1, 'iteration'))
C:\Users\quang\AppData\Local\Temp\ipykernel_8756\3282043579.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df.nlargest(1, 'iteration'))
C:\Users\quang\AppData\Local\Temp\ipykernel_8756\3282043579.py:61: DeprecationWarning: Dat